In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from collections import Counter
import pickle

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
X_train = pd.read_csv("data/processed/X_train.csv")
y_train = pd.read_csv("data/processed/y_train.csv")["class"]

with open("data/processed/feature_cols.pkl", "rb") as f:
    feature_cols = pickle.load(f)

print(f"Full training set: {len(X_train):,} rows x {len(feature_cols)} features")
print(f"Classes: {y_train.unique().tolist()}")

# Sample 100K rows for speed — EFS does not need the full dataset
SAMPLE = 100_000
if len(X_train) > SAMPLE:
    X_sample = X_train.sample(n=SAMPLE, random_state=42)
    y_sample = y_train.loc[X_sample.index]
    print(f"Using {SAMPLE:,} samples for EFS")
else:
    X_sample = X_train
    y_sample = y_train
    print(f"Using all {len(X_train):,} samples for EFS")

Full training set: 1,916,259 rows x 39 features
Classes: ['DDoS', 'Benign', 'Reconnaissance']
Using 100,000 samples for EFS


In [3]:
print("Running Method A: Random Forest (may take 1-3 mins)...")

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=feature_cols)
rf_top15 = importances.nlargest(15).index.tolist()

print("Top 15 features by Random Forest importance:")
for i, feat in enumerate(rf_top15, 1):
    print(f"  {i:2d}. {feat:<40} {importances[feat]:.4f}")

print("Method A done.")

Running Method A: Random Forest (may take 1-3 mins)...
Top 15 features by Random Forest importance:
   1. Header_Length                            0.0960
   2. Number                                   0.0782
   3. ack_flag_number                          0.0753
   4. TCP                                      0.0680
   5. IAT                                      0.0613
   6. Rate                                     0.0603
   7. ack_count                                0.0590
   8. Tot size                                 0.0529
   9. Tot sum                                  0.0442
  10. AVG                                      0.0402
  11. HTTPS                                    0.0377
  12. syn_flag_number                          0.0362
  13. Max                                      0.0352
  14. Time_To_Live                             0.0339
  15. syn_count                                0.0332
Method A done.


In [4]:
print("Running Method B: Chi-Square...")

chi_selector = SelectKBest(chi2, k=15)
chi_selector.fit(X_sample, y_sample)

chi_scores = pd.Series(chi_selector.scores_, index=feature_cols)
chi_top15 = chi_scores.nlargest(15).index.tolist()

print("Top 15 features by Chi-Square score:")
for i, feat in enumerate(chi_top15, 1):
    print(f"  {i:2d}. {feat:<40} {chi_scores[feat]:.4f}")

print("Method B done.")

Running Method B: Chi-Square...
Top 15 features by Chi-Square score:
   1. Number                                   48398.1629
   2. ICMP                                     41568.6873
   3. TCP                                      26364.0887
   4. ack_flag_number                          24325.3601
   5. HTTPS                                    22012.4659
   6. syn_flag_number                          13099.7709
   7. Header_Length                            10729.9282
   8. rst_flag_number                          10121.0197
   9. psh_flag_number                          5122.6975
  10. syn_count                                3675.0329
  11. UDP                                      3351.2252
  12. HTTP                                     2762.1555
  13. AVG                                      2332.3436
  14. Tot size                                 2332.3436
  15. ack_count                                2232.5508
Method B done.


In [5]:
print("Running Method C: Mutual Information (may take 2-5 mins)...")

mi_scores_arr = mutual_info_classif(X_sample, y_sample, random_state=42, n_jobs=-1)
mi_scores = pd.Series(mi_scores_arr, index=feature_cols)
mi_top15 = mi_scores.nlargest(15).index.tolist()

print("Top 15 features by Mutual Information score:")
for i, feat in enumerate(mi_top15, 1):
    print(f"  {i:2d}. {feat:<40} {mi_scores[feat]:.4f}")

print("Method C done.")

Running Method C: Mutual Information (may take 2-5 mins)...
Top 15 features by Mutual Information score:
   1. Tot sum                                  0.7599
   2. Header_Length                            0.7000
   3. Number                                   0.6407
   4. TCP                                      0.6127
   5. ack_flag_number                          0.5864
   6. Time_To_Live                             0.5761
   7. Tot size                                 0.5605
   8. AVG                                      0.5584
   9. IAT                                      0.5428
  10. ack_count                                0.5392
  11. Rate                                     0.5329
  12. Protocol Type                            0.5223
  13. Max                                      0.4909
  14. Variance                                 0.4581
  15. Std                                      0.4577
Method C done.


In [6]:
# Count votes across all three methods
votes = Counter(rf_top15 + chi_top15 + mi_top15)

# Build the vote table
all_nominated = set(rf_top15 + chi_top15 + mi_top15)
rows = []
for feat in sorted(all_nominated, key=lambda f: -votes[f]):
    rows.append({
        'Feature': feat,
        'Random Forest': 'YES' if feat in rf_top15 else '-',
        'Chi-Square':    'YES' if feat in chi_top15 else '-',
        'Mutual Info':   'YES' if feat in mi_top15 else '-',
        'Votes': votes[feat]
    })
vote_df = pd.DataFrame(rows)

print("=" * 70)
print("ENSEMBLE FEATURE SELECTION — VOTE TABLE")
print("=" * 70)
print(vote_df.to_string(index=False))

# Final set: features chosen by >= 3 of 3 methods
minimal_feature_set = sorted(
    [f for f, c in votes.items() if c >= 3],
    key=lambda f: -votes[f]
)

print(f"{'=' * 70}")
print(f"FINAL MINIMAL FEATURE SET: {len(minimal_feature_set)} features")
print(f"{'=' * 70}")
for i, feat in enumerate(minimal_feature_set, 1):
    print(f"  {i}. {feat}  (selected by {votes[feat]}/3 methods)")

ENSEMBLE FEATURE SELECTION — VOTE TABLE
        Feature Random Forest Chi-Square Mutual Info  Votes
      ack_count           YES        YES         YES      3
  Header_Length           YES        YES         YES      3
         Number           YES        YES         YES      3
       Tot size           YES        YES         YES      3
ack_flag_number           YES        YES         YES      3
            AVG           YES        YES         YES      3
            TCP           YES        YES         YES      3
syn_flag_number           YES        YES           -      2
            IAT           YES          -         YES      2
      syn_count           YES        YES           -      2
   Time_To_Live           YES          -         YES      2
        Tot sum           YES          -         YES      2
            Max           YES          -         YES      2
           Rate           YES          -         YES      2
          HTTPS           YES        YES           -      2


In [7]:
import os
os.makedirs("data/processed", exist_ok=True)

# Save as plain text (for your report and README)
with open("data/processed/minimal_feature_set.txt", "w") as f:
    for feat in minimal_feature_set:
        f.write(feat + "")

# Save as pickle (for use in the next notebooks)
with open("data/processed/minimal_feature_set.pkl", "wb") as f:
    pickle.dump(minimal_feature_set, f)

print(f"Saved {len(minimal_feature_set)} features to:")
print("  data/processed/minimal_feature_set.txt")
print("  data/processed/minimal_feature_set.pkl")
print(f"Final feature set: {minimal_feature_set}")

Saved 7 features to:
  data/processed/minimal_feature_set.txt
  data/processed/minimal_feature_set.pkl
Final feature set: ['Header_Length', 'Number', 'ack_flag_number', 'TCP', 'ack_count', 'Tot size', 'AVG']
